# Step 3 Part C: Combining option P&L + hedge P&L into true hedging error

So far we've only tracked the hedge position's P&L. The actual metric that matters is: option P&L + hedge P&L = total portfolio P&L. A PERFECT hedge would make this total flat (near zero) regardless of how the underlying moves. The 'hedging error' is how far from flat it actually is.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

train = pd.read_csv("btc_options_train.csv")
train["hour_bucket"] = pd.to_datetime(train["hour_bucket"])
train["sample_date"] = train["hour_bucket"].dt.date

def bs_delta(S, K, T_years, sigma, option_type, r=0.0):
    if T_years <= 0 or sigma <= 0:
        if option_type == "call":
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T_years) / (sigma * np.sqrt(T_years))
    return norm.cdf(d1) if option_type == "call" else norm.cdf(d1) - 1.0

counts = train.groupby(["symbol", "sample_date"]).size().sort_values(ascending=False)
proto_symbol, proto_date = counts.index[0]
episode = train[(train["symbol"] == proto_symbol) & (train["sample_date"] == proto_date)].sort_values("hour_bucket").reset_index(drop=True)
episode["T_years"] = episode["time_to_maturity_days"] / 365
episode["iv_decimal"] = episode["mark_iv"] / 100
episode["bs_delta"] = episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["iv_decimal"], row["type"]), axis=1
)
print(f"Episode: {proto_symbol} on {proto_date}, {len(episode)} steps")

## Option P&L

We are SHORT the option (we sold it to a client and are hedging our exposure -- the standard setup in deep hedging literature). So our option P&L is the NEGATIVE of the change in the option's own mark price each step: if the option's value goes up, we (the seller) lose that much, before considering the hedge.

We use `mid_price` (in BTC terms, per the original data) converted to USD using `underlying_price` at that time, since Deribit quotes option prices in BTC.

In [ ]:
# mid_price is in BTC; convert to USD
episode["option_mid_usd"] = episode["mid_price"] * episode["underlying_price"]
episode["option_pnl"] = -episode["option_mid_usd"].diff().fillna(0)  # we are SHORT the option
episode[["hour_bucket", "mid_price", "option_mid_usd", "option_pnl"]]

## Full simulation: hedge P&L, transaction costs, option P&L, and total hedging error

In [ ]:
def simulate_full_hedge(episode_df, target_delta_col="bs_delta", cost_rate_col="relative_spread"):
    n = len(episode_df)
    position = np.zeros(n)
    trade_size = np.zeros(n)
    transaction_cost = np.zeros(n)
    hedge_pnl = np.zeros(n)

    current_position = 0.0
    for i in range(n):
        target = episode_df[target_delta_col].iloc[i]
        trade = target - current_position
        trade_size[i] = trade

        spot = episode_df["underlying_price"].iloc[i]
        cost_rate = episode_df[cost_rate_col].iloc[i]
        transaction_cost[i] = abs(trade) * spot * (cost_rate / 2)

        if i > 0:
            prev_spot = episode_df["underlying_price"].iloc[i - 1]
            hedge_pnl[i] = current_position * (spot - prev_spot)

        current_position = target
        position[i] = current_position

    result = episode_df.copy()
    result["position"] = position
    result["trade_size"] = trade_size
    result["transaction_cost"] = transaction_cost
    result["hedge_pnl"] = hedge_pnl

    # THE KEY LINE: total portfolio P&L = option P&L + hedge P&L - transaction costs
    result["total_pnl"] = result["option_pnl"] + result["hedge_pnl"] - result["transaction_cost"]

    result["cumulative_cost"] = transaction_cost.cumsum()
    result["cumulative_hedge_pnl"] = hedge_pnl.cumsum()
    result["cumulative_option_pnl"] = result["option_pnl"].cumsum()
    result["cumulative_total_pnl"] = result["total_pnl"].cumsum()
    return result

sim = simulate_full_hedge(episode)
sim[["hour_bucket", "hedge_pnl", "option_pnl", "transaction_cost", "total_pnl", "cumulative_total_pnl"]]

## The key plot: cumulative total P&L

This is THE metric. A good hedge keeps this line close to flat/zero despite the underlying moving around. A bad hedge lets it drift far from zero. We also show hedge P&L and (negative) option P&L separately so you can see how well they offset each other -- that's the actual mechanism of hedging.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(sim["hour_bucket"], sim["cumulative_hedge_pnl"], label="Hedge P&L (cumulative)", marker="o")
axes[0].plot(sim["hour_bucket"], sim["cumulative_option_pnl"], label="Option P&L (cumulative, we're short)", marker="o")
axes[0].axhline(0, color="gray", linestyle="--")
axes[0].set_title("Hedge P&L vs Option P&L -- do they offset each other?")
axes[0].legend()

axes[1].plot(sim["hour_bucket"], sim["cumulative_total_pnl"], color="black", marker="o")
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_title("TOTAL portfolio P&L (option + hedge - costs) -- the actual hedging error")

plt.tight_layout()
plt.show()

print(f"Final total P&L (terminal hedging error): ${sim['total_pnl'].sum():.4f}")
print(f"Sum of |option P&L|: ${sim['option_pnl'].abs().sum():.4f}")
print(f"Sum of |hedge P&L|: ${sim['hedge_pnl'].abs().sum():.4f}")